# Tiền xử lý WikiText bằng Subword BPE

Notebook chuyển thể từ `tien_xu_ly.ipynb` của bạn. Giữ quy trình **tải dữ liệu → EDA → cleaning → tokenize → lưu ID → Dataset/DataLoader → kiểm tra**, nhưng thay Word-level bằng **Byte-level BPE**.

- Tokenizer chỉ học từ **train**.
- `validation` và `test` dùng đúng tokenizer train.
- Lưu kết quả trong `data/processed/bpe/`, không ghi đè Word-level.
- Token IDs được ghi trực tiếp vào `.npy` qua memory mapping để tiết kiệm RAM (quan trọng với WikiText-103).
- Chạy notebook khi thư mục làm việc hiện tại là thư mục gốc dự án, hoặc đặt `PROJECT_ROOT` phù hợp.


## 1. Cài thư viện

## 2. Import và cấu hình

In [ ]:
import re
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from tokenizers import ByteLevelBPETokenizer
from numpy.lib.format import open_memmap
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Đổi thành "wikitext-2-raw-v1" nếu muốn thử dữ liệu nhỏ trước.
DATASET_CONFIG = "wikitext-103-raw-v1"
BPE_VOCAB_SIZE = 10000
MIN_FREQ = 2
CONTEXT_LENGTH = 64
BATCH_SIZE = 32

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name in ("notebooks", "train"):
    PROJECT_ROOT = PROJECT_ROOT.parent
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "bpe"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SPECIAL_TOKENS = ["<pad>", "<unk>", "<eos>"]
print("Thư mục đầu ra:", OUTPUT_DIR)

## 3. Tải dữ liệu và kiểm tra

In [ ]:
dataset = load_dataset("Salesforce/wikitext", DATASET_CONFIG)
print(dataset)
for split in ["train", "validation", "test"]:
    print(f"{split}: {len(dataset[split]):,} dòng")
for i in range(10):
    print(i, repr(dataset["train"][i]["text"]))

## 4. EDA nhẹ (không tạo danh sách token khổng lồ)

In [ ]:
def analyze_split(split_data):
    total_lines = len(split_data)
    empty_lines = 0
    heading_lines = 0
    word_count_sum = 0
    max_words = 0
    special_char_lines = 0
    for row in tqdm(split_data, total=total_lines, desc="EDA"):
        text = row["text"]
        stripped = text.strip()
        if not stripped:
            empty_lines += 1
        if re.match(r"^=+.*=+$", stripped):
            heading_lines += 1
        words = len(text.split())
        word_count_sum += words
        max_words = max(max_words, words)
        if re.search(r"[^\w\s]", text, flags=re.UNICODE):
            special_char_lines += 1
    return {
        "Total lines": total_lines,
        "Empty lines": empty_lines,
        "Heading lines": heading_lines,
        "Average whitespace words": word_count_sum / max(total_lines, 1),
        "Max whitespace words": max_words,
        "Lines with punctuation/special chars": special_char_lines,
    }

eda_df = pd.DataFrame([
    {"Split": split, **analyze_split(dataset[split])}
    for split in ["train", "validation", "test"]
])
display(eda_df)
eda_df.to_csv(OUTPUT_DIR / "eda_summary.csv", index=False)

## 5. Cleaning giống notebook gốc

Chỉ loại dòng trống, gỡ dấu `=` của heading và chuẩn hóa khoảng trắng. Không lower-case, không bỏ stopword hoặc dấu câu. `<eos>` sẽ đánh dấu **kết thúc dòng không trống**, giống bản Word-level.

In [ ]:
def clean_text(text):
    text = text.strip()
    if not text:
        return ""
    text = re.sub(r"^\s*=+\s*(.*?)\s*=+\s*$", r"\1", text)
    return re.sub(r"\s+", " ", text)


def iter_clean_texts(split):
    for row in dataset[split]:
        text = clean_text(row["text"])
        if text:
            yield text

print("Ví dụ sau cleaning:")
from itertools import islice
for text in islice(iter_clean_texts("train"), 5):
    print(repr(text))

## 6. Huấn luyện Byte-level BPE trên TRAIN

**Không dùng validation/test để học merges/vocabulary.** Iterator giúp không tạo thêm danh sách `cleaned_data["train"]` lớn trong RAM.

In [ ]:
bpe_tokenizer = ByteLevelBPETokenizer()
bpe_tokenizer.train_from_iterator(
    iter_clean_texts("train"),
    vocab_size=BPE_VOCAB_SIZE,
    min_frequency=MIN_FREQ,
    special_tokens=SPECIAL_TOKENS,
)

vocab_size = bpe_tokenizer.get_vocab_size()
PAD_ID = bpe_tokenizer.token_to_id("<pad>")
UNK_ID = bpe_tokenizer.token_to_id("<unk>")
EOS_ID = bpe_tokenizer.token_to_id("<eos>")

bpe_tokenizer.save(str(OUTPUT_DIR / "tokenizer.json"))
print("Vocabulary:", vocab_size)
print("PAD/UNK/EOS:", PAD_ID, UNK_ID, EOS_ID)
assert all(x is not None for x in (PAD_ID, UNK_ID, EOS_ID))

## 7. Thử tokenizer và khả năng decode

In [ ]:
example = "The cats are unbelievable."
enc = bpe_tokenizer.encode(example, add_special_tokens=False)
print("Input:", example)
print("Subword tokens:", enc.tokens)
print("Token IDs:", enc.ids)
print("Decoded:", bpe_tokenizer.decode(enc.ids))

## 8. Encode từng split và ghi thẳng `.npy`

Hai lượt qua mỗi split: lượt 1 đếm token, lượt 2 ghi file. Chỉ giữ **mỗi dòng** trong RAM khi encode. Mặc định bổ sung một `<eos>` sau mỗi dòng không trống, giống notebook gốc. Nếu muốn thay đổi quy ước EOS, phải áp dụng đồng nhất ở cả Word và BPE.

In [ ]:
def encode_split_to_npy(split, path):
    total_ids = 0
    clean_lines = 0
    for text in tqdm(iter_clean_texts(split), desc=f"Count {split}"):
        total_ids += len(bpe_tokenizer.encode(text, add_special_tokens=False).ids) + 1
        clean_lines += 1

    if total_ids == 0:
        raise ValueError(f"{split} không có token")

    arr = open_memmap(str(path), mode="w+", dtype=np.int32, shape=(total_ids,))
    pos = 0
    unk_count = 0
    for text in tqdm(iter_clean_texts(split), total=clean_lines, desc=f"Encode {split}"):
        ids = bpe_tokenizer.encode(text, add_special_tokens=False).ids
        n = len(ids)
        if n:
            arr[pos:pos+n] = ids
            unk_count += ids.count(UNK_ID)
            pos += n
        arr[pos] = EOS_ID
        pos += 1

    assert pos == total_ids
    arr.flush()
    del arr
    return {
        "Split": split,
        "Clean lines": clean_lines,
        "Tokens": total_ids,
        "UNK Rate (%)": 100 * unk_count / total_ids,
        "Context Length": CONTEXT_LENGTH,
        "Vocabulary Size": vocab_size,
        "Batch Size": BATCH_SIZE,
    }

summary = []
for split, filename in [
    ("train", "train.npy"),
    ("validation", "valid.npy"),
    ("test", "test.npy"),
]:
    summary.append(encode_split_to_npy(split, OUTPUT_DIR / filename))

summary_df = pd.DataFrame(summary)
display(summary_df)

## 9. Lưu config và thống kê

In [ ]:
config = {
    "dataset": DATASET_CONFIG,
    "tokenizer": "byte_level_bpe",
    "tokenizer_file": "tokenizer.json",
    "vocab_size": vocab_size,
    "max_vocab_size": BPE_VOCAB_SIZE,
    "min_freq": MIN_FREQ,
    "context_length": CONTEXT_LENGTH,
    "batch_size": BATCH_SIZE,
    "pad_id": PAD_ID,
    "unk_id": UNK_ID,
    "eos_id": EOS_ID,
    "lowercase": False,
    "eos_policy": "after_each_nonempty_cleaned_line",
    "seed": SEED,
    "data_format": "npy_int32_memmap",
}
with open(OUTPUT_DIR / "config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)
summary_df.to_csv(OUTPUT_DIR / "data_summary.csv", index=False)
print("Đã lưu BPE tại:", OUTPUT_DIR)

## 10. Dataset / DataLoader thử nghiệm

Dùng NumPy memmap, chỉ chuyển từng mẫu thành `torch.long` khi lấy batch. Code này là kiểm tra độc lập; để mô hình dùng BPE, hãy cập nhật `src/data/dataset.py` cho nhánh `tokenizer_type="bpe"`.

In [ ]:
class LanguageModelingDataset(Dataset):
    def __init__(self, token_ids, context_length=64):
        self.tokens = token_ids
        self.context_length = context_length

    def __len__(self):
        return max(0, (len(self.tokens) - 1) // self.context_length)

    def __getitem__(self, idx):
        start = idx * self.context_length
        end = start + self.context_length
        x = np.array(self.tokens[start:end], dtype=np.int64, copy=True)
        y = np.array(self.tokens[start+1:end+1], dtype=np.int64, copy=True)
        return torch.from_numpy(x), torch.from_numpy(y)


def create_bpe_dataloaders(context_length=64, batch_size=32):
    files = ["train.npy", "valid.npy", "test.npy"]
    datasets_ = [LanguageModelingDataset(
        np.load(OUTPUT_DIR / filename, mmap_mode="r"), context_length
    ) for filename in files]
    loaders = [DataLoader(ds, batch_size=batch_size, shuffle=(i==0), num_workers=0)
               for i, ds in enumerate(datasets_)]
    return (*loaders, vocab_size)

train_loader, valid_loader, test_loader, vocab_size = create_bpe_dataloaders(
    CONTEXT_LENGTH, BATCH_SIZE
)
X, Y = next(iter(train_loader))
print("X:", X.shape, "Y:", Y.shape, "Vocabulary:", vocab_size)
assert X.shape == Y.shape
assert torch.equal(X[:, 1:], Y[:, :-1])
assert X.min() >= 0 and X.max() < vocab_size
assert Y.min() >= 0 and Y.max() < vocab_size
print("BPE Dataset/DataLoader: OK")

## 11. Kiểm tra decode lại token ID

Dùng tokenizer BPE để decode; **không dùng `idx2word` của Word-level**.

In [ ]:
sample_ids = X[0][:30].tolist()
print("IDs:", sample_ids)
print("Subwords:", [bpe_tokenizer.id_to_token(i) for i in sample_ids])
print("Decoded:", bpe_tokenizer.decode(sample_ids, skip_special_tokens=True))
print("Các file đã tạo:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(p.name, f"({p.stat().st_size / 1024**2:.2f} MB)")

## 12. Tích hợp với LSTM / CNN / Transformer

Trong notebook train, DataLoader cần đọc dữ liệu ở `data/processed/bpe/` thay vì bản Word-level, và `vocab_size` phải lấy từ BPE. **Phải khởi tạo mô hình mới** với vocab mới, không dùng checkpoint của Word-level.

Ví dụ sử dụng trực tiếp các DataLoader ở notebook này:

```python
train_loader, valid_loader, test_loader, vocab_size = create_bpe_dataloaders(64, 16)
EXPERIMENT_NAME = "bpe_c64"
```

Nếu dùng `src/data/dataset.py` chung của nhóm, bổ sung tham số `tokenizer_type="bpe"` để đọc `.npy` theo `mmap_mode="r"`; không cần tạo `word2idx.json` giả cho BPE. Khi inference, dùng `ByteLevelBPETokenizer.from_file(...)` để encode/decode.

**Lưu ý nghiên cứu:** Word-level PPL và BPE PPL tính trên các đơn vị token khác nhau, không nên so sánh số trực tiếp để kết luận tokenizer nào tốt hơn.
